# 🏥 Medical Knowledge Graph: Parallel Ingestion (Dual T4 GPU)

### 🚀 Optimization: Multi-GPU Log Streaming
This version streams logs from both GPUs so we can see exactly where it is stuck.

In [ ]:
# 1. Setup
!git clone https://github.com/marmor123/medical-knowledge-graph.git
%cd medical-knowledge-graph
!pip install -r requirements.txt
!pip install bitsandbytes>=0.46.1 json-repair

import subprocess
import threading
import time
import sys

def stream_output(pipe, prefix):
    for line in iter(pipe.readline, ''):
        sys.stdout.write(f"[{prefix}] {line}")
        sys.stdout.flush()

pdf_file = 'Pocket Medicine 9th Edition 2026.pdf'

# GPU 0: Pages 1-208
cmd0 = ["python", "-m", "src.etl.processor", "--pdf", pdf_file, "--limit", "208", "--out", "data/interim/raw_chunks_gpu0.jsonl"]

# GPU 1: Pages 209-416
cmd1 = ["python", "-m", "src.etl.processor", "--pdf", pdf_file, "--start", "209", "--limit", "208", "--out", "data/interim/raw_chunks_gpu1.jsonl"]

print("🚀 Launching Parallel Workers...")

p0 = subprocess.Popen(cmd0, env={**os.environ, "CUDA_VISIBLE_DEVICES": "0"}, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
p1 = subprocess.Popen(cmd1, env={**os.environ, "CUDA_VISIBLE_DEVICES": "1"}, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

t0 = threading.Thread(target=stream_output, args=(p0.stdout, "GPU_0"))
t1 = threading.Thread(target=stream_output, args=(p1.stdout, "GPU_1"))

t0.start(); t1.start()

while p0.poll() is None or p1.poll() is None:
    time.sleep(1)

print("✅ All workers finished.")